# Synthetic Hypergraph with Hy-MMSBM

This notebook uses `hypergraphx.generation.hy_mmsbm_sampling` to synthesize a hypergraph.

It creates:
- a random membership matrix `u` (nodes x communities)
- a simple affinity matrix `w` (communities x communities)
- one sampled synthetic hypergraph

Then it prints basic stats and saves a portable payload to `data/synthetic/`.

In [52]:
from pathlib import Path
import pickle

import numpy as np
from hypergraphx.generation.hy_mmsbm_sampling import HyMMSBMSampler

In [53]:
# Synthetic configuration
seed = 42
rng = np.random.default_rng(seed)

n_nodes = 1000
n_communities = 3

u_raw = np.ones(n_communities)

# Node-community memberships (rows sum to 1)
u = rng.dirichlet(alpha=u_raw, size=n_nodes)

# Community affinity (higher diagonal -> stronger within-community tendency)
w = np.array(
    [
        [0.5, 0.08, 0.08],
        [0.08, 0.5, 0.08],
        [0.08, 0.08, 0.5],
    ],
    dtype=float,
)

u.shape, w.shape

((1000, 3), (3, 3))

In [54]:
# Closest hard community assignment for each row of u
# (nearest one-hot vector in Euclidean distance; equivalent to argmax)
closest_comm = np.argmax(u, axis=1)

# Optional: one-hot version of the closest assignment
closest_onehot = np.eye(n_communities, dtype=int)[closest_comm]

print("closest_comm shape:", closest_comm.shape)
print("community counts:", np.bincount(closest_comm, minlength=n_communities))
closest_comm[:10]

closest_comm shape: (1000,)
community counts: [319 343 338]


array([0, 2, 1, 2, 0, 2, 2, 1, 0, 2])

In [56]:
# Sample two synthetic hypergraphs at different target average degrees
sampler = HyMMSBMSampler(
    u=u,
    w=w,
    seed=seed,
    max_hye_size=None,
    exact_dyadic_sampling=True,
    burn_in_steps=200,
    intermediate_steps=200,
)

target_avg_degs = [3, 15]
H_syn_by_deg = {}

for tgt in target_avg_degs:
    H_tmp = next(sampler.sample(avg_deg=tgt, allow_rescaling=True))
    H_syn_by_deg[tgt] = H_tmp
    avg_deg_emp = sum(len(e) for e in H_tmp.get_edges()) / len(H_tmp.get_nodes())
    print(f"target avg_deg={tgt} -> nodes={len(H_tmp.get_nodes())}, hyperedges={len(H_tmp.get_edges())}, empirical avg_deg={avg_deg_emp:.2f}")

target avg_deg=3 -> nodes=825, hyperedges=407, empirical avg_deg=2.54
target avg_deg=15 -> nodes=1000, hyperedges=1966, empirical avg_deg=11.82


In [57]:
# Quick check of achieved average degree for each synthetic hypergraph
for tgt, H_tmp in H_syn_by_deg.items():
    avg_deg_emp = sum(len(e) for e in H_tmp.get_edges()) / len(H_tmp.get_nodes())
    print(f"target={tgt}, empirical={avg_deg_emp:.2f}")

target=3, empirical=2.54
target=15, empirical=11.82


In [58]:
# Build simple synthetic labels from dominant community (for each sampled graph)
labels_full = np.argmax(u, axis=1)
payloads = {}

for tgt, H_tmp in H_syn_by_deg.items():
    nodes_sorted = sorted(H_tmp.get_nodes())
    labels = labels_full[np.asarray(nodes_sorted, dtype=int)]
    edge_list = [tuple(e) for e in H_tmp.get_edges()]

    payloads[tgt] = {
        "edge_list": edge_list,
        "labels": labels.tolist(),
        "u": u,
        "w": w,
        "seed": seed,
        "target_avg_deg": tgt,
    }

    print(f"target={tgt}: edge_list size={len(edge_list)}, labels shape={labels.shape}")

target=3: edge_list size=407, labels shape=(825,)
target=15: edge_list size=1966, labels shape=(1000,)


In [59]:
# Save both synthetic payloads
out_dir = Path("data") / "synthetic" / "hymmsbm"
out_dir.mkdir(parents=True, exist_ok=True)

for tgt, payload in payloads.items():
    out_path = out_dir / f"hy_mmsbm_avgdeg_{tgt}.pickle"
    with open(out_path, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved: {out_path}")

Saved: data/synthetic/hymmsbm/hy_mmsbm_avgdeg_3.pickle
Saved: data/synthetic/hymmsbm/hy_mmsbm_avgdeg_15.pickle
